# 02) Backtest → Export

Run the backtest via the backend API and export the results (metrics, equity curve, trades).\nGenerate charts and comprehensive analysis reports.

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import requests
import numpy as np

API_BASE = "http://localhost:8000"  # change if needed
EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse payload from notebook 01 or define again
# payload = {...}

## 1. Run backtest

Execute the backtest against the backend API. Results include metrics, equity curve, and trade history.

In [ ]:
result = requests.post(f"{API_BASE}/strategies/backtests", json=payload, timeout=300).json()
backtest_id = result.get("backtest_id", "unknown")
backtest_id

## 2. Inspect metrics

Key performance indicators from the backtest run.

In [ ]:
metrics = result.get("metrics", {})
metrics

## 3. Export JSON report

Save the full backtest report as JSON for archival and further analysis.

In [ ]:
json_path = EXPORT_DIR / f"backtest_report_{backtest_id}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
json_path

## 4. Export equity curve and trades to CSV

In [ ]:
equity_curve = metrics.get("equity_curve", [])
trades = metrics.get("trades", [])

df_eq = pd.DataFrame(equity_curve)
df_tr = pd.DataFrame(trades)

eq_path = EXPORT_DIR / f"equity_curve_{backtest_id}.csv"
tr_path = EXPORT_DIR / f"trades_{backtest_id}.csv"

df_eq.to_csv(eq_path, index=False)
df_tr.to_csv(tr_path, index=False)

eq_path, tr_path

## 5. Generate equity curve chart

In [ ]:
if equity_curve:
    df_eq["date"] = pd.to_datetime(df_eq["date"])
    df_eq = df_eq.sort_values("date")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(df_eq["date"], df_eq["equity"], color="#0EA5E9", linewidth=2, label="Equity")
    ax.fill_between(df_eq["date"], df_eq["equity"], alpha=0.1, color="#0EA5E9")
    ax.set_xlabel("Date")
    ax.set_ylabel("Portfolio Value (₹)")
    ax.set_title(f"Equity Curve - Backtest {backtest_id}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    chart_path = EXPORT_DIR / f"equity_curve_{backtest_id}.png"
    fig.savefig(chart_path, dpi=150, bbox_inches="tight")
    chart_path

## 6. Export config for reproducibility

In [ ]:
config_path = EXPORT_DIR / f"config_{backtest_id}.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
config_path

## 7. Export combined bundle

Creates a comprehensive bundle with all artifacts for integration.

In [ ]:
bundle = {
    "backtest_id": backtest_id,
    "backtest_result": result,
    "equity_curve_path": str(eq_path),
    "trades_path": str(tr_path),
    "report_path": str(json_path),
    "config_path": str(config_path),
    "exported_at": datetime.utcnow().isoformat() + "Z",
}

bundle_path = EXPORT_DIR / f"bundle_{backtest_id}.json"
with open(bundle_path, "w", encoding="utf-8") as f:
    json.dump(bundle, f, ensure_ascii=False, indent=2)
bundle_path